# Imports

In [1]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

Convert numeric age into groups

# Loading & Preprocessing the Dataset

In [2]:
def get_age_group(age):
  if age <= 12:
      return 0
  elif age <= 19:
      return 1
  elif age <= 35:
      return 2
  elif age <= 55:
      return 3
  else:
      return 4

## Dataset Class

In [3]:
class UTKFaceDataset(Dataset):
  def __init__(self, root_dir, transform=None):
    self.root_dir = root_dir
    self.transform = transform
    self.samples = []

    for file in os.listdir(root_dir):
      if not file.endswith(".jpg"):
        continue

      try:
        parts = file.split("_")
        age = int(parts[0])
        gender = int(parts[1])

        if age < 0 or age > 100:
          continue
        if gender not in [0,1]:
          continue

        age_group = get_age_group(age)

        self.samples.append((file, age_group, gender))

      except:
        continue # skip the bad files

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, idx):
    file, age_group, gender = self.samples[idx]
    img_path = os.path.join(self.root_dir, file)

    try:
      image = Image.open(img_path).convert('RGB')
    except:
      # fall back incase the image is corrupted
      return self.__getitem__(idx + 1 % len(self.samples))

    if self.transform:
      image = self.transform(image)

    return image, torch.tensor(age_group), torch.tensor(gender)




## preprocess

In [4]:
# transformation

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## Load Dataset UTKFace

In [5]:
pip install kaggle

In [6]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [7]:
!kaggle datasets download -d jangedoo/utkface-new

Dataset URL: https://www.kaggle.com/datasets/jangedoo/utkface-new
License(s): copyright-authors
100% 331M/331M [00:25<00:00, 13.8MB/s]



In [8]:
!unzip utkface-new.zip

Streaming output truncated to the last 5000 lines.
  inflating: utkface_aligned_cropped/crop_part1/34_1_0_20170109004755204.jpg.chip.jpg  
  inflating: utkface_aligned_cropped/crop_part1/34_1_0_20170111182452832.jpg.chip.jpg  
  inflating: utkface_aligned_cropped/crop_part1/34_1_1_20170103230340961.jpg.chip.jpg  
  inflating: utkface_aligned_cropped/crop_part1/34_1_1_20170104011329697.jpg.chip.jpg  
  inflating: utkface_aligned_cropped/crop_part1/34_1_1_20170104165020320.jpg.chip.jpg  
  inflating: utkface_aligned_cropped/crop_part1/34_1_1_20170108230211421.jpg.chip.jpg  
  inflating: utkface_aligned_cropped/crop_part1/34_1_2_20170104022134829.jpg.chip.jpg  
  inflating: utkface_aligned_cropped/crop_part1/34_1_2_20170104023010725.jpg.chip.jpg  
  inflating: utkface_aligned_cropped/crop_part1/34_1_2_20170104172537171.jpg.chip.jpg  
  inflating: utkface_aligned_cropped/crop_part1/34_1_2_20170104201443273.jpg.chip.jpg  
  inflating: utkface_aligned_cropped/crop_part1/34_1_2_20170104204327

In [9]:
dataset = UTKFaceDataset(root_dir='UTKFace', transform=transform)
print('done')

done


## Train and validation split

In [10]:
print(f'no.of sampes: {len(dataset)}')

no.of sampes: 23687


In [11]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

## Create DataLoaders

In [12]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)

In [13]:
# check

for images, age, gender in train_loader:
    print(images.shape, age.shape, gender.shape)
    break

torch.Size([8, 3, 224, 224]) torch.Size([8]) torch.Size([8])


# Model Building

In [14]:
# imports
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision.models import resnet18, ResNet18_Weights

## Model Architecture

In [15]:
class MultiTaskModel(nn.Module):
  def __init__(self, num_age_classes=5, num_gender_classes=2, num_emotion_classes=7):
    super(MultiTaskModel, self).__init__()

    # Load Pre-Trained Model
    self.backbone = resnet18(weights=ResNet18_Weights.DEFAULT)

    # Remove the fully connected layer
    in_features = self.backbone.fc.in_features
    self.backbone.fc = nn.Identity()

    # Heads
    self.age_head = nn.Sequential(
        nn.Linear(in_features, 128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128, num_age_classes)
    )
    self.gender_head = nn.Sequential(
        nn.Linear(in_features, 128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128, num_gender_classes)
    )
    self.emotion_head = nn.Sequential(
        nn.Linear(in_features, 128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128, num_emotion_classes)
    )

  def forward(self, x):
    features = self.backbone(x)

    age_out = self.age_head(features)
    gender_out = self.gender_head(features)
    emotion_out = self.emotion_head(features)

    return age_out, gender_out, emotion_out


## FER DATASET

In [16]:
# import pandas as pd
# import numpy as np
# from PIL import Image

# class FERDataset(Dataset):
#     def __init__(self, csv_file, transform=None):
#         self.data = pd.read_csv(csv_file)
#         self.transform = transform

#     def __len__(self):
#         return len(self.data)

#     def __getitem__(self, idx):
#         row = self.data.iloc[idx]

#         emotion = int(row['emotion'])
#         pixels = np.array(row['pixels'].split(), dtype=np.uint8)
#         image = pixels.reshape(48, 48)

#         image = Image.fromarray(image).convert("RGB")

#         if self.transform:
#             image = self.transform(image)

#         return image, torch.tensor(emotion)

In [17]:
!kaggle datasets download -d msambare/fer2013
!unzip fer2013.zip

Streaming output truncated to the last 5000 lines.
  inflating: train/sad/Training_65242339.jpg  
  inflating: train/sad/Training_65267116.jpg  
  inflating: train/sad/Training_65275626.jpg  
  inflating: train/sad/Training_6529266.jpg  
  inflating: train/sad/Training_65329617.jpg  
  inflating: train/sad/Training_65338712.jpg  
  inflating: train/sad/Training_65338797.jpg  
  inflating: train/sad/Training_65387162.jpg  
  inflating: train/sad/Training_65404494.jpg  
  inflating: train/sad/Training_65426218.jpg  
  inflating: train/sad/Training_65430136.jpg  
  inflating: train/sad/Training_65437377.jpg  
  inflating: train/sad/Training_6545735.jpg  
  inflating: train/sad/Training_65463385.jpg  
  inflating: train/sad/Training_65473985.jpg  
  inflating: train/sad/Training_65502829.jpg  
  inflating: train/sad/Training_65505359.jpg  
  inflating: train/sad/Training_65508578.jpg  
  inflating: train/sad/Training_65516023.jpg  
  inflating: train/sad/Training_65524027.jpg  
  inflating

In [18]:
# same Transform that we used before
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [19]:
from torchvision.datasets import ImageFolder

fer_train = ImageFolder(
    root='train',
    transform=transform
)

fer_val = ImageFolder(
    root='test',
    transform=transform
)

print('done')

done


In [20]:
from torch.utils.data import DataLoader

fer_train_loader = DataLoader(fer_train, batch_size=8, shuffle=True, num_workers=0)
fer_val_loader = DataLoader(fer_val, batch_size=8, shuffle=False, num_workers=0)

In [21]:
for img, label in fer_train_loader:
    print(img.shape)   # [32, 3, 224, 224]
    print(label.shape) # [32]
    print(label[:5])
    break

torch.Size([8, 3, 224, 224])
torch.Size([8])
tensor([3, 3, 5, 6, 5])


## Model Initailize

In [22]:
# model = MultiTaskModel()

# # Test forward pass
# x = torch.randn(8,3,224,224)
# age_pred, gender_pred = model(x)

# print(f'age out shape: {age_pred.shape}') # [8,5]
# print(f'gender out shape: {gender_pred.shape}') # [8,2]

# Training

In [23]:
# imports
import torch
import torch.nn as nn
import torch.optim as optim
from itertools import cycle

In [24]:
# device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [25]:
# model initialization
model = MultiTaskModel().to(device)

# loss functions
criterion_age = nn.CrossEntropyLoss()
criterion_gender = nn.CrossEntropyLoss()
criterion_emotion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Accuracy Function
def accuracy(preds, labels):
  preds = torch.argmax(preds, dim=1)
  return (preds == labels).float().mean()




Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 150MB/s]


In [26]:
# Training function
def train_one_epoch(model, utk_loader, fer_loader, optimizer, device):

  # put the model in train mode
  model.train()

  num_batch_size = 0
  total_loss = 0

  total_age_acc = 0
  total_gender_acc = 0
  total_emotion_acc = 0

  for (utk_batch, fer_batch) in zip(utk_loader, fer_loader):
    # for UTK batch
    utk_images, age_labels, gender_labels = utk_batch

    # moves to gpu
    utk_images = utk_images.to(device)
    age_labels = age_labels.to(device)
    gender_labels = gender_labels.to(device)

    # prediction (ignore emotion for now)
    age_out, gender_out,_ = model(utk_images)

    # Loss Calculation
    age_loss = criterion_age(age_out, age_labels)
    gender_loss = criterion_gender(gender_out, gender_labels)


    # for FER Batch
    fer_images, emotion_labels = fer_batch

    # move to gpu
    fer_images = fer_images.to(device)
    emotion_labels = emotion_labels.to(device)

    # prediction (ignore age & gender for now)
    _, _, emotion_out = model(fer_images)

    # Loss Calculation
    emotion_loss = criterion_emotion(emotion_out, emotion_labels)

    # Combined Loss
    total_batch_loss = age_loss + gender_loss + 0.5 * emotion_loss

    # backward & weight updation
    optimizer.zero_grad()
    total_batch_loss.backward()
    optimizer.step()

    total_loss += total_batch_loss.item()

    total_age_acc += accuracy(age_out, age_labels).item()
    total_gender_acc += accuracy(gender_out, gender_labels).item()
    total_emotion_acc += accuracy(emotion_out, emotion_labels).item()

    # counter
    num_batch_size +=1

  return (
      total_loss / num_batch_size,
      total_age_acc / num_batch_size,
      total_gender_acc / num_batch_size,
      total_emotion_acc / num_batch_size
  )

In [27]:
# # validation Function
# def validate(model, utk_loader, fer_loader, device):
#   # put the model in evaluation mode
#   model.eval()

#   total_loss = 0

#   total_age_acc = 0
#   total_gender_acc = 0
#   total_emotion_acc = 0

#   with torch.no_grad():
#     for (utk_batch, fer_batch) in zip(utk_loader, fer_loader):

#       # For UTK Batch
#       images, age_labels, gender_labels = utk_batch

#       # moves to gpu
#       images = images.to(device)
#       age_labels = age_labels.to(device)
#       gender_labels = gender_labels.to(device)

#       # prediction
#       age_out, gender_out, _ = model(images)

#       # Loss Calculation
#       age_loss = criterion_age(age_out, age_labels)
#       gender_loss = criterion_gender(gender_out, gender_labels)

#       loss = age_loss + gender_loss

#       total_loss += loss.item()

#       total_age_acc = accuracy(age_out, age_labels).item()
#       total_gender_acc = accuracy(gender_out, gender_labels).item()

#       # For FER Batch
#       images, emotion_labels = fer_batch

#       # move to gpu
#       images = images.to(device)
#       emotion_labels = emotion_labels.to(device)

#       # prediction
#       _, _, emotion_out = model(images)

#       # Loss Calculation
#       emotion_loss = criterion_emotion(emotion_out, emotion_labels)

#       total_loss += emotion_loss.item()

#       total_emotion_acc = accuracy(emotion_out, emotion_labels).item()


#   return (
#       total_loss / len(utk_loader),
#       total_age_acc / len(utk_loader),
#       total_gender_acc / len(utk_loader),
#       total_emotion_acc / len(utk_loader)
#   )

In [28]:
# UTK Validation loop
def validate_utk(model, loader, device):
  model.eval()

  total_loss = 0
  total_age_accuracy = 0
  total_gender_accuracy = 0

  with torch.no_grad():
    for images, age_labels, gender_labels in loader:

      images = images.to(device)
      age_labels = age_labels.to(device)
      gender_labels = gender_labels.to(device)

      age_out, gender_out, _ = model(images)

      age_loss = criterion_age(age_out, age_labels)
      gender_loss = criterion_gender(gender_out, gender_labels)
      loss = age_loss + gender_loss

      total_loss += loss.item()
      total_age_accuracy += accuracy(age_out, age_labels).item()
      total_gender_accuracy += accuracy(gender_out, gender_labels).item()

  return (
      total_loss / len(loader),
      total_age_accuracy / len(loader),
      total_gender_accuracy / len(loader)
  )

In [29]:
# FER Validation loop
def validate_fer(model, loader, device):
  model.eval()

  total_loss = 0
  total_emotion_accuracy = 0

  with torch.no_grad():
    for images, emotion_labels in loader:
      images = images.to(device)
      emotion_labels = emotion_labels.to(device)

      _, _, emotion_out = model(images)

      emotion_loss = criterion_emotion(emotion_out, emotion_labels)

      total_loss += emotion_loss.item()
      total_emotion_accuracy += accuracy(emotion_out, emotion_labels).item()

  return (
      total_loss / len(loader),
      total_emotion_accuracy / len(loader)
  )

## Training and validation Loop

In [30]:
epochs = 16
patience = 3

best_val_loss = float('inf')
counter = 0

for epoch in range(epochs):

  train_loss, train_age_acc, train_gender_acc, train_emotion_acc = train_one_epoch(model, train_loader, fer_train_loader, optimizer, device)
  utk_loss, val_age_acc, val_gender_acc = validate_utk(model, val_loader, device)
  fer_loss, val_emotion_acc = validate_fer(model, fer_val_loader, device)

  val_loss = (utk_loss + fer_loss) / 2

  print(f"\nEpoch: {epoch+1}/{epochs}")
  print(f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')
  print(f'Train Age Acc: {train_age_acc} | vall Age Acc: {val_age_acc}')
  print(f"Train Gender Acc: {train_gender_acc} | Val Gender Acc: {val_gender_acc}")
  print(f"Train Emotion Acc: {train_emotion_acc} | Val Emotion Acc {val_emotion_acc}")

  # Early Stoping + Save best model
  if val_loss < best_val_loss:
    best_val_loss = val_loss
    torch.save(model.state_dict(), 'best_model.pth')
    print('best model saved ')
    counter = 0
  else:
    counter += 1
    if counter > patience :
      print("Early stopping Triggered")
      break


Epoch: 1/16
Train Loss: 2.3175 | Val Loss: 1.3501
Train Age Acc: 0.5976044744718624 | vall Age Acc: 0.6637858347386172
Train Gender Acc: 0.7913043478361511 | Val Gender Acc: 0.8497048903878583
Train Emotion Acc: 0.30656395103419165 | Val Emotion Acc 0.3834910913140312
best model saved 

Epoch: 2/16
Train Loss: 1.9184 | Val Loss: 1.2255
Train Age Acc: 0.6773849725723267 | vall Age Acc: 0.6817032040472175
Train Gender Acc: 0.855846348670325 | Val Gender Acc: 0.8836424957841484
Train Emotion Acc: 0.3980054875474884 | Val Emotion Acc 0.45197661469933187
best model saved 

Epoch: 3/16
Train Loss: 1.7709 | Val Loss: 1.1702
Train Age Acc: 0.7000738708416386 | vall Age Acc: 0.7200674536256324
Train Gender Acc: 0.8730476994512453 | Val Gender Acc: 0.883220910623946
Train Emotion Acc: 0.4388455044322499 | Val Emotion Acc 0.4696547884187082
best model saved 

Epoch: 4/16
Train Loss: 1.6700 | Val Loss: 1.0974
Train Age Acc: 0.7180139299383038 | vall Age Acc: 0.726602023608769
Train Gender Acc: 0.

Epoch: 13/16
- Train Loss: 1.2157 | Val Loss: 0.9433
- Train Age Acc: 0.7922541156706804 | vall Age Acc: 0.7580101180438449
- Train Gender Acc: 0.9373364288779742 | Val Gender Acc: 0.9169477234401349
- Train Emotion Acc: 0.5862178134233854 | Val Emotion Acc 0.5921492204899778
- best model saved

# Inference

In [32]:
model = MultiTaskModel().to(device)
model.load_state_dict(torch.load('/content/best_model.pth'))
print('model loaded succesfully')

model loaded succesfully
